# Preprocessing dan Unsupervised Learning

Notebook ini membahas transformasi data, reduksi dimensi, faktorisasi matriks, manifold learning, dan clustering dengan API scikit-learn modern.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_digits, make_blobs, make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer, QuantileTransformer
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.decomposition import PCA, NMF
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import accuracy_score, adjusted_rand_score, silhouette_score
from scipy.cluster.hierarchy import dendrogram, ward

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Scaling dan pencegahan data leakage

In [ ]:
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=.25, random_state=RANDOM_STATE, stratify=cancer.target)

minmax = MinMaxScaler().fit(X_train)
X_train_mm, X_test_mm = minmax.transform(X_train), minmax.transform(X_test)
standard = StandardScaler().fit(X_train)
X_train_std, X_test_std = standard.transform(X_train), standard.transform(X_test)
quantile = QuantileTransformer(output_distribution="normal", random_state=RANDOM_STATE).fit(X_train)
X_train_q, X_test_q = quantile.transform(X_train), quantile.transform(X_test)
normalizer = Normalizer().fit(X_train)
X_train_norm, X_test_norm = normalizer.transform(X_train), normalizer.transform(X_test)

print("train/test:", X_train.shape, X_test.shape)
print("MinMax range:", X_train_mm.min(), X_train_mm.max())


## 2. Pengaruh preprocessing pada SVM

In [ ]:
models = {
    "tanpa scaling": SVC(C=10),
    "MinMax": make_pipeline(MinMaxScaler(), SVC(C=10)),
    "Standard": make_pipeline(StandardScaler(), SVC(C=10)),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "test accuracy:", round(model.score(X_test, y_test), 3))


## 3. PCA sebagai reduksi dimensi

In [ ]:
pca = make_pipeline(StandardScaler(), PCA(n_components=2))
X_pca = pca.fit_transform(cancer.data)
print("Original shape:", cancer.data.shape)
print("PCA shape:", X_pca.shape)
plt.scatter(X_pca[:,0], X_pca[:,1], c=cancer.target, cmap="coolwarm", s=15)
plt.title("PCA breast cancer")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()


## 4. PCA dan ekstraksi fitur pada citra

LFW digunakan jika sudah tersedia secara lokal. Jika tidak tersedia, notebook memakai digits sebagai data citra offline sehingga kode tetap dapat dijalankan.

In [ ]:
try:
    from sklearn.datasets import fetch_lfw_people
    people = fetch_lfw_people(min_faces_per_person=20, resize=.7, download_if_missing=False)
    X_img, y_img, image_name = people.data, people.target, "LFW faces"
except Exception:
    digits = load_digits()
    X_img, y_img, image_name = digits.data / 16.0, digits.target, "digits fallback"

Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    X_img, y_img, test_size=.25, random_state=RANDOM_STATE, stratify=y_img)
base_knn = KNeighborsClassifier(n_neighbors=1).fit(Xi_train, yi_train)
img_model = make_pipeline(StandardScaler(), PCA(n_components=min(100, Xi_train.shape[1]), whiten=True, random_state=RANDOM_STATE),
                           KNeighborsClassifier(n_neighbors=1))
img_model.fit(Xi_train, yi_train)
print("Image source:", image_name)
print("KNN raw/PCA accuracy:", round(base_knn.score(Xi_test, yi_test),3), round(img_model.score(Xi_test,yi_test),3))


## 5. NMF pada sinyal sintetis dan citra

In [ ]:
S = np.abs(np.random.RandomState(RANDOM_STATE).normal(size=(2000, 3)))
A = np.random.RandomState(RANDOM_STATE).uniform(size=(100, 3))
X_signal = S @ A.T
nmf = NMF(n_components=3, init="nndsvda", random_state=RANDOM_STATE, max_iter=1000)
S_recovered = nmf.fit_transform(X_signal)
print("Measurements/recovered:", X_signal.shape, S_recovered.shape)

X_img_nonnegative = np.maximum(X_img, 0)
nmf_img = NMF(n_components=min(15, X_img_nonnegative.shape[1]), init="nndsvda",
             random_state=RANDOM_STATE, max_iter=1000)
W_img = nmf_img.fit_transform(X_img_nonnegative)
print("NMF image representation:", W_img.shape, nmf_img.components_.shape)


## 6. t-SNE setelah PCA awal

In [ ]:
digits = load_digits()
digits_small = StandardScaler().fit_transform(digits.data)
digits_pca = PCA(n_components=30, random_state=RANDOM_STATE).fit_transform(digits_small)
digits_tsne = TSNE(n_components=2, init="pca", learning_rate="auto",
                  perplexity=30, random_state=RANDOM_STATE).fit_transform(digits_pca)
plt.scatter(digits_tsne[:,0], digits_tsne[:,1], c=digits.target, cmap="tab10", s=8)
plt.title("t-SNE digits after PCA")
plt.show()


## 7. K-Means dan failure cases

In [ ]:
X_blob, y_blob = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=RANDOM_STATE)
kmeans = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE).fit(X_blob)
print("K-Means ARI:", round(adjusted_rand_score(y_blob, kmeans.labels_),3))

X_moon, y_moon = make_moons(n_samples=200, noise=.05, random_state=RANDOM_STATE)
moon_kmeans = KMeans(n_clusters=2, n_init=20, random_state=RANDOM_STATE).fit(X_moon)
print("K-Means moons silhouette:", round(silhouette_score(X_moon, moon_kmeans.labels_),3))

# Vector quantization: setiap titik direpresentasikan oleh centroid terdekat.
distance_features = kmeans.transform(X_blob)
print("Distance feature shape:", distance_features.shape)


## 8. Agglomerative clustering dan dendrogram

In [ ]:
agg = AgglomerativeClustering(n_clusters=3, linkage="ward").fit(X_blob)
print("Agglomerative ARI:", round(adjusted_rand_score(y_blob, agg.labels_),3))
linkage_matrix = ward(X_blob)
plt.figure(figsize=(12,4))
dendrogram(linkage_matrix, truncate_mode="level", p=3)
plt.title("Dendrogram")
plt.show()


## 9. DBSCAN, noise, dan evaluasi clustering

In [ ]:
dbscan = DBSCAN(eps=.3, min_samples=5).fit(X_moon)
labels_db = dbscan.labels_
mask = labels_db != -1
print("DBSCAN clusters:", len(set(labels_db)) - (1 if -1 in labels_db else 0))
print("Noise points:", int(np.sum(labels_db == -1)))
if len(set(labels_db[mask])) > 1:
    print("Silhouette without noise:", round(silhouette_score(X_moon[mask], labels_db[mask]),3))

# ARI tidak terpengaruh oleh pertukaran nomor label.
print("ARI ground truth:", round(adjusted_rand_score(y_moon, labels_db),3))
for eps in [.1, .2, .3, .4, .5, .7]:
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_moon)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    print(f"eps={eps}: clusters={n_clusters}, noise={np.sum(labels==-1)}")


## 10. Clustering pada data citra

In [ ]:
X_img_pca = PCA(n_components=min(30, X_img.shape[1]), random_state=RANDOM_STATE).fit_transform(StandardScaler().fit_transform(X_img))
km_img = KMeans(n_clusters=10, n_init=20, random_state=RANDOM_STATE).fit(X_img_pca)
agg_img = AgglomerativeClustering(n_clusters=10, linkage="ward").fit(X_img_pca)
print("K-Means cluster sizes:", np.bincount(km_img.labels_).tolist())
print("Agglomerative cluster sizes:", np.bincount(agg_img.labels_).tolist())
print("K-Means versus agglomerative ARI:", round(adjusted_rand_score(km_img.labels_, agg_img.labels_),3))


## Ringkasan

Notebook ini membahas seluruh alur preprocessing dan unsupervised learning. Label tidak dipakai untuk membentuk cluster, melainkan hanya untuk evaluasi setelah proses selesai. Gunakan pipeline dan beberapa nilai parameter agar tidak terjadi data leakage dan agar hasil lebih stabil.